# Eigendecomposition, SVD, and PCA — Week 2.

You may use `np.linalg.eigh` and `np.linalg.svd` here. Implementing a numerically
stable eigensolver is a numerical-analysis course, not an ML course. What you are
implementing is everything *around* those calls: centering, ordering, sign
conventions, variance accounting, reconstruction, and the equivalence between the
covariance-eigendecomposition and SVD routes to the same answer.

The single most valuable thing in this file is `test_pca_two_ways_agree`. When
you can explain *why* the eigenvectors of X^T X are the right singular vectors of
X, you understand PCA at the level an interviewer is probing for.

In [1]:
from __future__ import annotations
from dataclasses import dataclass, field

import numpy as np
from numpy.typing import NDArray

Matrix = NDArray[np.float64]
Vector = NDArray[np.float64]

In [2]:
def checkSqMat(A: Matrix) -> bool:
    if A.shape[0] != A.shape[1]:
        return False
    else:
        return True

In [3]:
def transpose(A: Matrix) -> Matrix:
    """Return the transpose of A without calling `.T`."""
    A_rows = 1 if len(A.shape) == 1 else A.shape[0]
    A_cols = A.shape[0] if len(A.shape) == 1 else A.shape[1]
    AT = np.zeros((A_cols, A_rows))
    for i in range (A_cols):
        for j in range (A_rows):
            if len(A.shape) == 1:
                AT[i, j] = A[i]
            else:
                AT[i, j] = A[j, i]
    return AT

In [4]:
def isSymmetric(A: Matrix) -> bool:
    if checkSqMat(A):
        for i in range(A.shape[0]):
            for j in range(A.shape[0]):
                if i != j:
                    if A[i][j] != A[j][i]:
                        return False
        return True
    else:
        return False

In [5]:
def eigen_decomposition(A: Matrix) -> tuple[Vector, Matrix]:
    """Eigendecomposition of a symmetric matrix, sorted by descending eigenvalue.

    NumPy's `eigh` returns eigenvalues in *ascending* order. Every downstream
    convention in ML assumes descending. Reversing it is a one-liner and
    forgetting it is a bug you will make exactly once.

    Args:
        A: Symmetric matrix of shape (n, n).

    Returns:
        (eigenvalues, eigenvectors) where eigenvalues has shape (n,) sorted
        descending, and eigenvectors has shape (n, n) with eigenvector i as
        *column* i.

    Raises:
        ValueError: if A is not square or not symmetric.
    """
    if not isSymmetric(A):
        raise ValueError("ValueError: if A is not square or not symmetric.")
    eigenvalues, eigenvectors = np.linalg.eigh(A)
    return eigenvalues[::-1], eigenvectors[:, ::-1]

In [6]:
a = np.array([
    [4, 0, -1],
    [0, 5, 2],
    [-1, 2, 7]
])
eigen_decomposition(a)

(array([8.40267883, 4.31603086, 3.28129031]),
 array([[-0.19216509,  0.7154086 ,  0.6717612 ],
        [ 0.49727948,  0.6611152 , -0.56181831],
        [ 0.84604119, -0.2260912 ,  0.48280128]]))

In [7]:
def power_iteration(A: Matrix, num_iters: int = 1000, tol: float = 1e-10) -> tuple[float, Vector]:
    """Find the dominant eigenvalue/eigenvector by repeated multiplication.

    Start from a random unit vector, multiply by A, normalize, repeat. It
    converges to the eigenvector with the largest absolute eigenvalue, at a rate
    governed by the ratio of the top two eigenvalues.

    Worth implementing for two reasons: it makes eigenvectors feel like an
    attractor rather than an algebraic definition, and it is the conceptual
    ancestor of PageRank and of the power method inside randomized SVD.

    Returns:
        (eigenvalue, eigenvector) with the eigenvector normalized to unit length.
    """
    v = np.random.randn(A.shape[0])
    v = v / np.linalg.norm(v)
    Av = np.zeros(A.shape[0])
    for i in range (num_iters):
        Av = np.matmul (A,v)
        l2 = np.linalg.norm(Av)
        v_norm = Av / l2
        if np.linalg.norm(v_norm - v) < tol:
            break
        v = v_norm
    eigenvalue = np.dot(v, np.matmul(A, v))
    return eigenvalue, v

In [8]:
a = np.array([
    [4, 0, -1],
    [0, 5, 2],
    [-1, 2, 7]
])
power_iteration(a, 10000)

(np.float64(8.40267882952119), array([ 0.19216509, -0.49727948, -0.84604119]))

In [9]:
def svd(A: Matrix, full_matrices: bool = False) -> tuple[Matrix, Vector, Matrix]:
    """Singular value decomposition: A = U @ diag(S) @ Vt.

    Wrap `np.linalg.svd` and guarantee the conventions the rest of this module
    relies on: singular values non-negative and sorted descending.

    Geometric reading, which is the one to have in an interview: *every* linear
    map is a rotation (Vt), then an axis-aligned scaling (S), then another
    rotation (U). There is nothing else a matrix can do.

    Returns:
        (U, S, Vt) with S of shape (min(m, n),).
    """
    U, S, Vt = np.linalg.svd(A, full_matrices=full_matrices)
    return U, S, Vt

In [10]:
a = np.array([
    [4, 0, -1],
    [-1, 2, 7]
])
U, S, Vt = svd(a)
print(U)
print(S)
print(Vt)

[[-0.26501331  0.96424475]
 [ 0.96424475  0.26501331]]
[7.5513736  3.73855009]
[[-0.26807017  0.25538261  0.92893385]
 [ 0.96079111  0.14177331  0.23828715]]


In [11]:
def truncated_svd(A: Matrix, k: int) -> tuple[Matrix, Vector, Matrix]:
    """Keep only the top k singular triplets.

    Args:
        A: shape (m, n).
        k: Number of components. Must satisfy 1 <= k <= min(m, n).

    Returns:
        (U_k, S_k, Vt_k) with shapes (m, k), (k,), (k, n).
    """
    if k > min(A.shape[0], a.shape[1]):
        raise ValueError("Value of k should be 1 <= k <= min(m, n)")
    U, S, Vt = svd(A)
    return U[:k,], S[:k,], Vt[:k,]

In [12]:
A = np.array([
    [1, 2, 3, 4, 5, 6, 7, 8],
    [9, 10, 11, 12, 13, 14, 15, 16],
    [17, 18, 19, 20, 21, 22, 23, 24],
    [25, 26, 27, 28, 29, 30, 31, 32],
    [33, 34, 35, 36, 37, 38, 39, 40]
])
#U, S, Vt = truncated_svd(a, 2)
truncated_svd(A, 2)

(array([[-0.08905037, -0.76946087,  0.07619844,  0.26819655,  0.56768337],
        [-0.24073058, -0.49198454,  0.26938876, -0.66845094, -0.42497416]]),
 array([148.63152196,   6.97643742]),
 array([[-0.30605442, -0.31925521, -0.332456  , -0.3456568 , -0.35885759,
         -0.37205838, -0.38525918, -0.39845997],
        [ 0.56832857,  0.41459092,  0.26085328,  0.10711564, -0.046622  ,
         -0.20035965, -0.35409729, -0.50783493]]))

In [ ]:
def low_rank_approximation(A: Matrix, k: int) -> Matrix:
    """Best rank-k approximation of A in the Frobenius norm.

    The Eckart-Young theorem says truncated SVD is *optimal* — no rank-k matrix
    is closer. That is a strong statement and it is why SVD underpins
    compression, denoising, and (loosely) why LoRA's low-rank update is a
    defensible parameterization rather than an arbitrary one.

    Returns:
        Shape (m, n), rank at most k.
    """
    raise NotImplementedError("Week 2")